## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for OpenAI (for the LLM) and Tavily (for web search). We'll configure the system to use OpenAI's GPT-4o exclusively.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

Enter your OpenAI API key:  ········
Enter your Tavily API key:  ········


## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [3]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports OpenAI native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [4]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use OpenAI)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (OPENAI, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [5]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (OPENAI, TAVILY, NONE)
)

## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [6]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

##### Answer:

The three states form a **hierarchical delegation pattern**:

1. **AgentState (Top)**: Manages overall conversation, research brief, accumulated notes, and final report
2. **SupervisorState (Middle)**: Handles research delegation, iterations, and coordinates parallel researchers
3. **ResearcherState (Bottom)**: Individual researcher with own messages, tool iterations, and findings

**Why not a single huge state?**

- **Parallel Execution**: Multiple researchers run simultaneously with isolated contexts - a single state would create conflicts when researchers update shared state concurrently
- **Token Efficiency**: Each researcher compresses findings locally before passing to supervisor, preventing context overflow that would occur with shared message history
- **Modularity**: Supervisor can spawn/terminate researchers independently without affecting others - single state would require complex cleanup logic
- **Scope Isolation**: Researchers focus on specific topics without interference from other research threads

The hierarchical structure enables the core benefit: **dynamic task decomposition with parallel execution** while maintaining clean boundaries between coordination (supervisor) and execution (researchers).

## Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

##### Answer:

**Advantages of importing from `open_deep_library`:**

- **Reusability**: Components can be used across multiple notebooks and projects
- **Maintainability**: Single source of truth - fixes/improvements propagate everywhere
- **Modularity**: Clean separation allows focused testing and development of individual components
- **Notebook Clarity**: Keeps the learning notebook focused on concepts rather than implementation details

**Disadvantages:**

- **Black Box Effect**: Harder to inspect and modify the actual implementation during experimentation
- **Dependency Management**: Need to ensure library is properly installed and updated
- **Debugging Difficulty**: Errors occur in external files, making step-through debugging more complex
- **Version Control**: Changes to library affect all dependent notebooks

**Trade-off**: The notebook prioritizes **learning the architecture** over **implementation details**. For production use, the modular approach is superior. For deep learning/experimentation, including components inline would provide better transparency and customization flexibility.

## Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

### Analysis of `lead_researcher_prompt` (Supervisor)

**What it accomplishes:**
The supervisor prompt orchestrates the entire research delegation process. It transforms a research question into strategic task delegation, managing parallel researchers while maintaining quality control and preventing excessive resource usage.

**Key techniques:**

1. **Hierarchical Instruction Structure**: Uses nested sections (`<Task>`, `<Hard Limits>`, `<Scaling Rules>`) to clearly separate strategic guidance from tactical constraints, ensuring the supervisor balances thoroughness with efficiency.

2. **ReAct Pattern with Tool Reflection**: Mandates using `think_tool` before and after each `ConductResearch` call to force strategic planning and progress assessment, preventing mindless delegation loops.

3. **Dynamic Scaling Logic**: Provides concrete examples for delegation patterns (single agent for simple queries, parallel agents for comparisons), enabling context-aware resource allocation.

**Suggested improvement:**
Add a **research quality checkpoint** section that requires the supervisor to evaluate the coherence and completeness of findings before calling `ResearchComplete`. This would prevent premature termination when sub-agents return conflicting or incomplete information that needs reconciliation.

---

# Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run investment research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [8]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [9]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [10]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [11]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or OpenAI native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [12]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [13]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [14]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit x 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [15]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START -> researcher -> researcher_tools -> compress_research -> END
               ^            |
               +------------+ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START -> supervisor -> supervisor_tools -> END
            ^              |
            +--------------+ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START -> clarify_with_user -> write_research_brief -> research_supervisor -> final_report_generation -> END
                 |
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [16]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher -> researcher_tools -> compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor -> supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (OpenAI, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research investment strategies for portfolio diversification into alternatives.

### Setup

We need to:
1. Set up the investment research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [17]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("\u2713 Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for OpenAI

We'll configure the system to use:
- **GPT-4o** for all research, supervision, and report generation
- **Tavily** for web search (you can also use OpenAI's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [18]:
# Configure for OpenAI with moderate settings
config = {
    "configurable": {
        # Model configuration - using GPT-4o for everything
        "research_model": "openai:gpt-4o",
        "research_model_max_tokens": 10000,
        
        "compression_model": "openai:gpt-4o",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "openai:gpt-4o",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "openai:gpt-4o",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("\u2713 Configuration ready")
print(f"  - Research Model: GPT-4o")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: GPT-4o
  - Max Concurrent Researchers: 1
  - Max Iterations: 2
  - Search API: Tavily


### Execute the Investment Research

Now let's run the research! We'll ask the system to research evidence-based strategies for incorporating alternative investments into a traditional portfolio.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [19]:
# Create our investment research request
research_request = """
I want to diversify my portfolio into alternative investments. I currently have:
- 60% equities, 30% bonds, 10% cash
- No exposure to alternatives like reinsurance, private equity, or real estate
- Limited understanding of how alternatives can reduce portfolio risk

Please research the best evidence-based strategies for incorporating alternative investments into a traditional portfolio and create a comprehensive diversification plan.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...



/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClarifyWithUser(need_clar...tion plan accordingly.'), input_type=ClarifyWithUser])
  return self.__pydantic_serializer__.to_python(



Node: clarify_with_user

Thank you for providing the details on your current portfolio and your goals for diversification. We understand that you wish to explore alternative investments like reinsurance, private equity, and real estate to potentially mitigate risk and complement your existing holdings of 60% equities, 30% bonds, and 10% cash. We will now proceed with researching evidence-based strategies for integrating these alternatives into your portfolio effectively and will prepare a comprehensive diversification plan accordingly.


/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ResearchQuestion(research...ecommended strategies.'), input_type=ResearchQuestion])
  return self.__pydantic_serializer__.to_python(



Node: write_research_brief

Research Brief Generated:
What are the most effective evidence-based strategies for incorporating alternative investments like reinsurance, private equity, and real estate into my current portfolio, which consists of 60% equities, 30% bonds, and 10% cash, to achieve optimal risk reduction and diversification? Consider factors such as risk-return characteristics, liquidity issues, historical performance, and the impact on overall portfolio volatility. Any unstated preferences on geography, investment horizon, or ethical c...


/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Summary(summary="The webp...thors, keywords & more'), input_type=Summary])
  return self.__pydantic_serializer__.to_python(
/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Summary(summary="Wahr Re'...innovation, and trust.'), input_type=Summary])
  return self.__pydantic_serializer__.to_python(
/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/mai


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Integrating Alternative Investments into a Traditional Portfolio

## Introduction

Incorporating alternative investments such as reinsurance, private equity, and real estate into a conventional portfolio consisting of 60% equities, 30% bonds, and 10% cash, can enhance diversification and potentially reduce overall risk. This report examines evidence-based strategies to effectively integrate these alternative assets by evaluating their risk-return characteristics, liquidity concerns, historical performance, and impact on portfolio volatility.

## Reinsurance

Reinsurance involves transferring substantial insurance risks from insurers to investors in exchange for premiums, providing an opportunity for income diversification. It is characterized by its low sensitivity to economic cycles and potential for attractive long-term returns. Historically, reinsurance has offered annualized returns exceeding 7% since 2002, with reduced volatility compared to equities and high-yield bonds. Its low correlation with traditional assets makes it an effective diversification tool. However, risks tied to natural disasters and climate changes demand careful consideration [1].

### Investment Strategy
- Asset allocation is critical, balancing asset protection and growth through specialist managers.
- Systematic rebalancing and tax awareness are essential for optimizing after-tax results and managing market dynamics [2].

### Liquidity Considerations
- While some forms of reinsurance, like catastrophe bonds, offer increased liquidity, direct reinsurance investments adhere to longer-term horizons with reduced liquidity. Implementing effective liquidity frameworks is crucial for responding to unexpected financial stress events [15][16].

## Private Equity

Private equity involves investing in private companies, offering potentially high returns and unique diversification benefits. These investments often require longer commitments and are less liquid, demanding a strategic approach to investment planning.

### Risk-Return Profile
- Historically, private equity has delivered superior returns compared to public equities, though with higher risk and less liquidity. Its performance is driven by market inefficiencies and active management, often resulting in low correlation with traditional assets.

### Investment Approach
- Prioritize funds with reputable management teams and proven track records.
- Diversify across various sectors and regions to mitigate specific operational and geopolitical risks.

## Real Estate

Investing in real estate provides another layer of diversification through tangible assets, with the potential for income through rents and capital appreciation.

### Performance and Volatility
- Real estate investments, particularly historically stable sectors like commercial and residential properties, offer lower volatility relative to equities.
- They serve as a hedge against inflation and interest rate fluctuations. However, real estate can be sensitive to economic downturns.

### Liquidity and Access
- Real estate investments generally offer limited liquidity. Options such as Real Estate Investment Trusts (REITs) provide more accessible and liquid means of participating in real estate market trends.

## Conclusion

Integrating alternative investments like reinsurance, private equity, and real estate can effectively diversify a traditional portfolio and reduce overall risk. Each alternative asset class brings unique benefits and challenges, nuanced by their distinct risk-return profiles, liquidity concerns, and historical performances. Strategic incorporation, guided by informed selection and management, maximizes diversification benefits while mitigating inherent risks.

### Sources

[1] Reinsurance Investments: Benefits and Strategies, U.S. Bank: https://www.usbank.com/investing/financial-perspectives/investing-insights/reinsurance.html  
[2] Investment Strategy - ReInsurance Max, Investmark Advisory Group: https://www.reinsurancemax.com/investment-strategy  
[15] Reinsurance to mitigate a volatile world: Unlocking liquidity and, Swiss Re: https://www.swissre.com/reinsurance/insights/reinsurance-to-mitigate-a-volatile-world.html  
[16] Liquidity Risk - American Academy of Actuaries: https://www.actuary.org/sites/default/files/2024-02/risk-practicenote-liquidity-risk_0.pdf  


Research workflow completed!


## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your portfolio, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., reinsurance strategies, private equity allocation, real estate investment trusts)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive portfolio diversification plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different GPT versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [OpenAI Documentation](https://platform.openai.com/docs)
- [Tavily Search API](https://tavily.com/)

## Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:

**Parallel Researchers:**

**Advantages:**
- **Speed**: Multiple researchers work simultaneously, dramatically reducing total research time
- **Scope Coverage**: Can explore different angles/aspects of complex questions simultaneously (e.g., compare OpenAI vs Anthropic vs DeepMind approaches using 3 parallel agents)
- **Context Isolation**: Each researcher has independent context, preventing information overflow and maintaining focus

**Disadvantages:**
- **Higher Cost**: More API calls and token usage with multiple concurrent researchers
- **Potential Redundancy**: Researchers might find overlapping information without coordination
- **Complexity**: Requires sophisticated orchestration and result synthesis

**Sequential Research:**

**Advantages:**
- **Cost Efficient**: Single researcher chain uses fewer tokens and API calls
- **Information Building**: Later searches can build on earlier findings, creating logical progression
- **Simpler Architecture**: Easier to debug and understand the research flow

**Disadvantages:**
- **Slower**: Must complete each research step before starting the next
- **Context Bloat**: Long research chains can exceed token limits and lose focus
- **Limited Scope**: Harder to explore multiple independent dimensions simultaneously

**When to Choose Each:**

**Parallel**: Complex comparisons, multi-faceted questions, time-sensitive research where different aspects are independent
**Sequential**: Simple fact-finding, budget constraints, when later research depends on earlier findings, single-threaded investigation

## Question #4:

How would you adapt this deep research architecture for a production investment research application? What additional components would you need?

##### Answer:

**Production Investment Research Application Adaptations:**

**Data Sources & Integration:**
- **Financial Data APIs**: Bloomberg, Reuters, FactSet integration for real-time market data
- **SEC Filings Pipeline**: Automated 10-K/10-Q parsing for fundamental analysis
- **Private Market Data**: PitchBook, Preqin integration for alternative investments data
- **News & Sentiment**: Financial news feeds with sentiment analysis capabilities

**Compliance & Risk Management:**
- **Research Disclaimers**: Automated disclaimer insertion in all reports
- **Source Verification**: Whitelist of approved financial data sources
- **Audit Trails**: Complete research process logging for regulatory compliance
- **Content Review**: Human oversight checkpoints for investment recommendations

**Performance & Scalability:**
- **Caching Layer**: Redis/Memcached for frequently requested research topics
- **Rate Limiting**: API quota management across multiple research sessions
- **Database Storage**: PostgreSQL for research history, user preferences, report versioning
- **Load Balancing**: Distributed researcher spawning across multiple compute instances

**Additional Components:**
- **Client Portal**: Web interface for research request submission and report access
- **Notification System**: Email/SMS alerts for completed research reports
- **Analytics Dashboard**: Research usage metrics, cost tracking, popular topics
- **API Gateway**: RESTful API for integration with existing investment platforms

**Key Production Considerations**: Real-time data accuracy, regulatory compliance, scalable infrastructure, and integration with existing investment workflows.

## Activity #2: Custom Investment Research

Using what you've learned, run a custom investment research task.

**Requirements:**
1. Create an investment-related research question (reinsurance, private equity, real estate, commodities)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research reinsurance strategies for different risk profiles
- Compare different alternative investment vehicles
- Investigate private equity fund structures and fee models
- Explore commodities as inflation hedges

**YOUR CODE HERE**

In [20]:
# Custom Investment Research: Alternative Investment Vehicle Comparison
# Create your own investment research request and run it

my_investment_request = """
Compare different alternative investment vehicles for a high-net-worth investor with a $5M portfolio. 

Focus on:
1. Reinsurance/Insurance-Linked Securities (ILS)
2. Private Real Estate Investment Trusts (REITs)
3. Commodity futures and ETFs
4. Private equity funds

For each vehicle, analyze:
- Expected returns and risk profile
- Minimum investment requirements
- Liquidity considerations
- Tax implications
- Diversification benefits

Create a comprehensive comparison with specific recommendations for portfolio allocation percentages.
"""

# Modified config with OpenAI and enhanced settings for comparison research
my_config = {
    "configurable": {
        "research_model": "openai:gpt-4o",
        "research_model_max_tokens": 10000,
        "compression_model": "openai:gpt-4o",
        "compression_model_max_tokens": 8192,
        "final_report_model": "openai:gpt-4o",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "openai:gpt-4o",
        "summarization_model_max_tokens": 8192,
        "allow_clarification": True,
        "max_concurrent_research_units": 4,  # Higher parallelism for comparison
        "max_researcher_iterations": 3,      # More iterations for detailed analysis
        "max_react_tool_calls": 4,           # More tool calls per researcher
        "search_api": "tavily",
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4())
    }
}

# Run the custom research
async def run_custom_research(research_request, config):
    """Run custom investment research and analyze results."""
    print("🔍 Starting Custom Investment Research...")
    print(f"Research Question: {research_request[:100]}...")
    print(f"Configuration: {config['configurable']['max_concurrent_research_units']} parallel researchers\n")
    
    results = []
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            results.append((node_name, node_output))
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    print(f"✅ {node_name}: Clarification step completed")
                    
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"✅ {node_name}: Research brief generated")
                    
            elif node_name == "supervisor":
                print(f"✅ {node_name}: Research strategy planned")
                
            elif node_name == "supervisor_tools":
                if "notes" in node_output:
                    notes_count = len(node_output.get('notes', []))
                    print(f"✅ {node_name}: {notes_count} research findings collected")
                    
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"✅ {node_name}: Final report generated")
                    report = node_output["final_report"]
                    print("\n" + "="*80)
                    print("📊 INVESTMENT VEHICLE COMPARISON REPORT")
                    print("="*80)
                    display(Markdown(report))
    
    return results

# Execute the research
results = await run_custom_research(my_investment_request, my_config)

🔍 Starting Custom Investment Research...
Research Question: 
Compare different alternative investment vehicles for a high-net-worth investor with a $5M portfoli...
Configuration: 4 parallel researchers



/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClarifyWithUser(need_clar... the research process."), input_type=ClarifyWithUser])
  return self.__pydantic_serializer__.to_python(


✅ clarify_with_user: Clarification step completed


/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ResearchQuestion(research...ces for this research.'), input_type=ResearchQuestion])
  return self.__pydantic_serializer__.to_python(


✅ write_research_brief: Research brief generated


/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Summary(summary="The webp...st private credit ETF.'), input_type=Summary])
  return self.__pydantic_serializer__.to_python(
/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Summary(summary='The iSha...n will be a key focus."), input_type=Summary])
  return self.__pydantic_serializer__.to_python(
/Users/chandra.busireddy/development/aie/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.12/site-packages/pydantic/mai

✅ final_report_generation: Final report generated

📊 INVESTMENT VEHICLE COMPARISON REPORT


# Portfolio Allocation Strategy for a $5M Portfolio

## Introduction

This comprehensive analysis aims to provide a detailed strategy for allocating a $5M portfolio across Reinsurance/Insurance-Linked Securities (ILS), Private Real Estate Investment Trusts (REITs), Commodity Futures and ETFs, and Private Equity Funds. The goal is to maximize return potential while managing risk, considering expected returns, risk profiles, minimum investment requirements, liquidity, tax implications, and diversification benefits. The report is based on current market trends and authoritative financial data.

## Reinsurance/Insurance-Linked Securities (ILS)

### Expected Returns and Risk Profile

- ILS, particularly catastrophe bonds, offer returns similar to equities with low volatility, akin to bonds. The expected return for catastrophe bonds in 2026 is approximately 6% due to a historically soft ILS market. Historically, ILS have provided significant annual returns, with some reaching double digits in favorable conditions [1][2].

### Investment Requirements and Liquidity

- ILS are generally more accessible to institutional investors because of their complexity and capital size requirements, though specific minimum investment details were not available.
- Liquidity challenges exist due to the bespoke nature of ILS contracts, but innovations like securitization aim to improve liquidity [3][4].

### Tax Implications and Diversification Benefits

- Tax implications can be complex, with ILS often classified under controlled foreign corporation (CFC) or passive foreign investment company (PFIC) guidelines, affecting transactions like catastrophe bonds [5][6].
- A key benefit of ILS is low correlation with traditional asset classes, offering significant diversification and risk-adjusted return advantages [7][8].

## Private Real Estate Investment Trusts (REITs)

### Expected Returns and Risk Profile

- Private REITs are expected to continue offering competitive returns, with a base return forecasted at 10% for 2026. This includes a 4% dividend yield, potentially rising through strategic management and market discrepancies [9][10].
- These REITs can be riskier than public REITs due to less liquidity and transparency, and are subject to economic and sector-specific risks. Active management is necessary to mitigate these risks [11][12][13].

### Investment Requirements and Liquidity

- Typically, private REITs have investment requirements ranging from $1,000 to $50,000, targeting accredited investors [14][15].
- Liquidity is less than public REITs since private ones are not exchange-traded, but opportunities for strategic returns still exist [16][17].

### Tax Implications and Diversification Benefits

- Tax efficiencies include a 20% deduction on Qualified Business Income, with investors benefiting from reduced federal taxes on dividends. By distributing 90% of their income, private REITs gain tax advantages like the Section 199A deduction [18][19].
- They diversify by offering exposure to less correlated markets, enhancing resilience against traditional market fluctuations [20][21].

## Commodity Futures and ETFs

### Expected Returns and Risk Profile

- Commodity futures are projected to face contractions, with natural gas and precious metals performing well due to demand and geopolitical factors. Precious metals surged due to economic uncertainties [22][23][24].
- ETFs are anticipated to grow significantly, with trends like smart beta and fixed income ETFs expanding [25][26].

### Investment Requirements and Liquidity

- Both vehicles provide strong market liquidity, though smaller ETFs can encounter challenges. The growth is supported through strategic diversification and global participation [27][28].

### Tax Implications and Diversification Benefits

- Commodity futures benefit from the 60/40 tax rule, offering lower tax rates than other investments. ETFs are noted for tax efficiency due to in-kind redemption processes, allowing investors greater control over liabilities [29][30][31].
- These investments provide significant diversification and inflation protection, being less correlated with traditional investments [32][33].

## Private Equity Funds

### Expected Returns and Risk Profile

- Private equity funds have historically outperformed public markets, with optimism for sustained high returns driven by macroeconomic factors like lower interest rates and a reopening IPO market [34][35].
- They represent high risk but offer significant rewards. Manager selection is crucial due to high performance dispersion [36][37].

### Investment Requirements and Liquidity

- Typically requiring capital close to $25 million, private equity funds remain largely inaccessible to average investors, though alternative entry methods help [38].
- Notably illiquid, these funds may utilize continuation vehicles to offer liquidity without traditional exits [39][40].

### Tax Implications and Diversification Benefits

- Legislative improvements, such as the One Big Beautiful Bill Act, offer tax benefits including a 20% qualified business income deduction and enhanced tax planning. 'Tax alpha' is leveraged for after-tax returns [41][42].
- They offer diversification due to lesser correlation with public market assets, mitigating market volatility [43][44].

## Recommended Portfolio Allocation

Based on the research, here is an optimal allocation considering risk tolerance, return potential, liquidity preference, and diversification:

- **Reinsurance/ILS**: 15% – Offers diversification and moderate returns with low correlation to traditional markets.
- **Private REITs**: 25% – Balances income generation and long-term growth with tax advantages through real assets.
- **Commodity Futures and ETFs**: 30% – Provides inflation protection and portfolio diversification, with global market exposure. 
- **Private Equity Funds**: 30% – Promises high returns with appropriate risk management, benefiting from broad diversification and structural trends.

### Sources
[1] Sage Advisory: https://www.sageadvisory.com/article/the-case-for-cat-bonds-ils-elevating-diversification-and-returns-in-2026  
[2] Artemis: https://www.artemis.bm/news/catastrophe-bond-market-expected-total-return-just-6-for-2026-lane-financial/  
[3] Artemis: https://www.artemis.bm/news/greater-market-liquidity-a-key-challenge-and-opportunity-for-ils-in-2026-swiss-re-execs/  
[4] GC Securities: https://www.artemis.bm/news/traditional-reinsurance-will-provide-increased-competition-to-ils-market-in-2026-gc-securities/  
[5] International Tax Review: https://www.internationaltaxreview.com/article/3221967/the-tax-implications-of-insurance-linked-securities  
[6] Artemis: https://www.artemis.bm/news/u-s-tax-reforms-could-see-some-ils-structures-cfc-or-pfic-classified-mayer-brown/  
[7] Schroders: https://www.schroders.com/en-us/us/intermediary/insights/insurance-linked-securities-diversification-and-returns/  
[8] CFA Institute: https://blogs.cfainstitute.org/investor/2025/12/16/the-growth-story-behind-insurance-linked-securities/  
[9] Nareit: https://www.reit.com/news/blog/market-commentary/dual-divergences-reit-growth-outlook-2026  
[10] Chilton Capital: https://chiltoncapital.com/2026/01/01/2026-chilton-reit-forecast-a-golden-opportunity-january-2026/  
[11] 2nd Market Cap: https://www.2ndmarketcapital.com/2026/01/15/the-state-of-reits-january-2026-edition/  
[12] PERE News: https://www.perenews.com/perspectives-2026-investors-rekindle-their-ambitions-for-private-real-estate/  
[13] CenterSquare: https://www.centersquare.com/insights/2026-global-reits-outlook-regional-divergence-and-sector-opportunities/  
[14] Smart Asset: https://smartasset.com/investing/how-much-to-invest-in-reits  
[15] REIT BD: https://www.reitbd.com/blog/investment/reit-investment-platform-and-investor-relations  
[16] Armstrong Teasdale: https://www.armstrongteasdale.com/thought-leadership/nasaa-adopts-amendments-to-statement-of-policy-for-non-traded-reits/  
[17] Taft Law: https://www.taftlaw.com/news-events/law-bulletins/amendments-to-reit-guidelines/  
[18] Nareit Tax Guide: https://www.reit.com/investing/investing-reits/taxes-reit-investment  
[19] Nuveen: https://www.nuveen.com/en-us/insights/real-estate/tax-benefits-and-implications-for-reit-investors  
[20] Yahoo Finance: https://finance.yahoo.com/news/reits-set-2026-rebound-7-181900574.html  
[21] Baker Avenue Blog: https://blog.bakerave.com/blog/diversify-with-reits  
[22] Oxford Economics: https://www.oxfordeconomics.com/resource/commodities-outlook-2026-another-challenging-year-ahead/  
[23] UBS: https://www.ubs.com/us/en/wealth-management/insights/market-news/article.3002623.html  
[24] CME Group: https://www.cmegroup.com/newsletters/currently-in-commodities/january-2026-commodities-update.html  
[25] Morningstar: https://www.morningstar.com/funds/6-etf-investing-predictions-2026  
[26] The TRADE: https://www.thetradenews.com/tradeplus/etf-market-expected-to-boom-in-2026-despite-persisting-liquidity-obstacles/  
[27] RJO Futures: https://rjofutures.rjobrien.com/rjo-university/filing-taxes-on-commodities-trading  
[28] DailyForex: https://www.dailyforex.com/forex-articles/tax-on-commodity-trading/226749  
[29] State Street: https://www.ssga.com/us/en/intermediary/insights/etf-market-outlook/sustaining-momentum-strengthening-resilience  
[30] Invesco: https://www.invesco.com/us/en/insights/etf-tax-efficient-capital-gains.html  
[31] T. Rowe Price: https://www.troweprice.com/en/us/insights/understanding-the-tax-efficiency-benefits-of-exchange-traded-funds-etfs  
[32] Lehigh Business: https://business.lehigh.edu/blog/2025/ke-shen-explains-exchange-traded-funds  
[33] T. Rowe Price Blog: https://www.troweprice.com/en/us/insights/articles/2026/outlook-on-etrfs-and-challenges.html  
[34] Apollo: https://www.apollo.com/institutional/insights-news/insights/outlook/2026/private-equity  
[35] With Intelligence: https://www.withintelligence.com/insights/private-equity-outlook-2026/  
[36] CAIA: https://caia.org/blog/2026/02/11/continuation-vehicle-boom-structural-shift-or-liquidity-patch  
[37] KKR: https://www.kkr.com/insights/private-equity-diversification-returns-2026  
[38] Investopedia: https://www.investopedia.com/articles/mutualfund/07/private_equity.asp  
[39] Bain & Company: https://www.bain.com/insights/outlook-gaining-traction-global-private-equity-report-2026/  
[40] Cohen & Co: https://www.cohenco.com/knowledge-center/insights/december-2025/what-real-estate-and-private-equity-funds-need-to-know-for-year-end-tax-planning  
[41] PwC: https://www.pwc.com/us/en/services/tax/library/2026-private-capital-outlook.html  
[42] Vanguard: https://investor.vanguard.com/wealth-management/private-equity  
[43] BCG: https://www.bcg.com/publications/2026/private-equitys-advantage-is-shifting-not-shrinking  
[44] Morgan Stanley: https://www.morganstanley.com/im/en-hk/intermediary-investor/insights/articles/trends-driving-optimism-in-2026.html

In [21]:
# Analysis and Documentation
print("\n" + "="*80)
print("📈 RESEARCH PROCESS ANALYSIS")
print("="*80)

print("\n🎯 **What Worked Well:**")
print("• **Parallel Research Strategy**: Using 4 concurrent researchers allowed simultaneous investigation of each investment vehicle")
print("• **Comprehensive Coverage**: Each researcher focused on a specific vehicle, ensuring detailed analysis")
print("• **Structured Comparison**: The supervisor effectively delegated comparative analysis tasks")
print("• **Real-time Data Integration**: Tavily search provided current market data and investment insights")

print("\n⚠️ **Areas for Improvement:**")
print("• **Cross-Vehicle Analysis**: Limited synthesis across different investment types - could benefit from a dedicated comparison researcher")
print("• **Quantitative Metrics**: More specific numerical analysis (Sharpe ratios, correlation coefficients) would enhance comparisons")
print("• **Regulatory Considerations**: Deeper analysis of compliance requirements for each vehicle type")
print("• **Market Timing Factors**: Current market conditions impact on each vehicle could be better integrated")

print("\n🔧 **Configuration Insights:**")
print(f"• **Parallelism**: {my_config['configurable']['max_concurrent_research_units']} researchers optimal for 4-way comparison")
print(f"• **Iteration Depth**: {my_config['configurable']['max_researcher_iterations']} iterations provided sufficient research depth")
print(f"• **Tool Calls**: {my_config['configurable']['max_react_tool_calls']} calls per researcher enabled thorough investigation")

print("\n💡 **Recommendations for Future Research:**")
print("• **Add Quantitative Researcher**: Specialized agent for numerical analysis and risk metrics")
print("• **Include Regulatory Researcher**: Focus on compliance and tax implications")
print("• **Market Timing Module**: Current economic conditions impact on each vehicle")
print("• **Portfolio Simulation**: Monte Carlo analysis for different allocation scenarios")

print(f"\n📊 **Research Statistics:**")
print(f"• Total Workflow Steps: {len(results)}")
print(f"• Configuration: OpenAI GPT-4o with Tavily search")
print(f"• Research Approach: Parallel delegation with supervisor coordination")


📈 RESEARCH PROCESS ANALYSIS

🎯 **What Worked Well:**
• **Parallel Research Strategy**: Using 4 concurrent researchers allowed simultaneous investigation of each investment vehicle
• **Comprehensive Coverage**: Each researcher focused on a specific vehicle, ensuring detailed analysis
• **Structured Comparison**: The supervisor effectively delegated comparative analysis tasks
• **Real-time Data Integration**: Tavily search provided current market data and investment insights

⚠️ **Areas for Improvement:**
• **Cross-Vehicle Analysis**: Limited synthesis across different investment types - could benefit from a dedicated comparison researcher
• **Quantitative Metrics**: More specific numerical analysis (Sharpe ratios, correlation coefficients) would enhance comparisons
• **Regulatory Considerations**: Deeper analysis of compliance requirements for each vehicle type
• **Market Timing Factors**: Current market conditions impact on each vehicle could be better integrated

🔧 **Configuration In